# فاحص بيئة بيان | Bayan Runtime Doctor

**إعداد وتقديم | Prepared and delivered by:** ميعاد المري · Meaad Al-Marri  
**المسار | Lane:** 🟢 Core · **الوقت | Time:** 10–15 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/almiyead-rgb/bayan-applied-nlp-course/blob/develop/notebooks/00_runtime_doctor.ipynb)

شغّل الخلايا من الأعلى إلى الأسفل. نجاح CPU كافٍ؛ GPU ليس شرطًا.  
Run the cells top to bottom. CPU success is sufficient; a GPU is not required.


## ماذا سيفحص؟ | What it checks

- إصدار Python ونظام التشغيل.
- توفر المكتبات الأساسية الموجودة عادةً في Colab.
- سلامة UTF-8 والنص العربي.
- الجهاز الفعلي: CPU أو CUDA GPU.
- الذاكرة والمساحة المتاحة تقريبًا.
- الوصول إلى المواقع المطلوبة، بوصفه تحذيرًا لا شرط نجاح.
- إنشاء `runtime_report.json` بلا أسرار أو بيانات شخصية.


In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import platform
import shutil
import sys
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

COURSE = "Bayan Applied NLP"
AUTHOR = "Meaad Al-Marri"

def package_version(name: str) -> str:
    try:
        return version(name)
    except PackageNotFoundError:
        return "not-installed"

def gib(value: int) -> float:
    return round(value / (1024 ** 3), 2)

try:
    in_colab = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    in_colab = False
torch_version = package_version("torch")
cuda_available = False
gpu_name = None

if torch_version != "not-installed":
    import torch
    cuda_available = bool(torch.cuda.is_available())
    if cuda_available:
        gpu_name = torch.cuda.get_device_name(0)

disk = shutil.disk_usage("/")
report = {
    "course": COURSE,
    "author": AUTHOR,
    "checked_at_utc": datetime.now(timezone.utc).isoformat(),
    "in_colab": in_colab,
    "python": platform.python_version(),
    "platform": platform.platform(),
    "device": "cuda" if cuda_available else "cpu",
    "gpu_name": gpu_name,
    "disk_free_gib": gib(disk.free),
    "packages": {
        "numpy": package_version("numpy"),
        "pandas": package_version("pandas"),
        "scikit-learn": package_version("scikit-learn"),
        "torch": torch_version,
    },
}

print(json.dumps(report, ensure_ascii=False, indent=2))


In [ ]:
# Core checks: these determine readiness. Network and GPU do not.
arabic_sample = "مرحبًا بكم في مشروع بيان"
core_checks = {
    "python_3_10_or_newer": sys.version_info >= (3, 10),
    "utf8_round_trip": arabic_sample.encode("utf-8").decode("utf-8") == arabic_sample,
    "numpy_available": report["packages"]["numpy"] != "not-installed",
    "pandas_available": report["packages"]["pandas"] != "not-installed",
    "sklearn_available": report["packages"]["scikit-learn"] != "not-installed",
    "torch_available": report["packages"]["torch"] != "not-installed",
    "disk_has_1_gib_free": report["disk_free_gib"] >= 1.0,
}

for name, passed in core_checks.items():
    print(("✅" if passed else "❌"), name)

BAYAN_ENV_READY = all(core_checks.values())
print("\nBAYAN_ENV_READY =", BAYAN_ENV_READY)
if not BAYAN_ENV_READY:
    raise RuntimeError("Environment check failed. Copy the failed check names and use the setup troubleshooting guide.")


In [ ]:
# Connectivity checks are warnings because classroom networks may filter sites.
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

sites = {
    "github": "https://github.com",
    "pypi": "https://pypi.org",
    "huggingface": "https://huggingface.co",
}
connectivity = {}
for name, url in sites.items():
    try:
        request = Request(url, headers={"User-Agent": "Bayan-Course-Runtime-Doctor/1.0"})
        with urlopen(request, timeout=10) as response:
            connectivity[name] = 200 <= response.status < 400
    except (HTTPError, URLError, TimeoutError, OSError) as exc:
        connectivity[name] = False
        print(f"⚠️ {name}: {type(exc).__name__}")

report["connectivity"] = connectivity
for name, passed in connectivity.items():
    print(("✅" if passed else "⚠️"), name)

print("\nA connectivity warning does not change BAYAN_ENV_READY.")


In [ ]:
# Save a small, safe report. It contains no password, token, email, or Drive path.
report["core_checks"] = core_checks
report["bayan_env_ready"] = BAYAN_ENV_READY
report_path = Path("runtime_report.json")
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved:", report_path.resolve())
print("Size:", report_path.stat().st_size, "bytes")


## اختياري: Google Drive | Optional

لا تشغّل الخلية التالية إلا بعد قراءة صلاحية الوصول. تركيب Drive يسمح للكود بالوصول إلى ملفات Drive. لا تمنح الصلاحية لدفتر غير موثوق.


In [ ]:
MOUNT_DRIVE = False  # Change to True only when you choose to mount Drive.

if MOUNT_DRIVE:
    if not in_colab:
        raise RuntimeError("Drive mounting is available in Google Colab, not this runtime.")
    from google.colab import drive
    drive.mount("/content/drive")
    recovery_dir = Path("/content/drive/MyDrive/bayan-nlp/recovery")
    recovery_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(report_path, recovery_dir / report_path.name)
    print("Copied report to:", recovery_dir)
else:
    print("Drive not mounted. This is normal for the first readiness check.")


## النتيجة التالية | Next step

إذا ظهر `BAYAN_ENV_READY = True`:

1. احفظ نسخة notebook في Drive.
2. احتفظ بملف `runtime_report.json`.
3. أكمل [إعداد GitHub](../docs/setup/github.md).
4. لا تقلق إن كان الجهاز CPU؛ هذا يحقق شرط الجاهزية.

If readiness is false, copy the failed check names and open the [troubleshooting guide](../docs/setup/troubleshooting.md).
